# 02_visualizacion_y_limpieza.ipynb

**Responsable:** Gabriel

**Etapas:**
* c) Preprocesamiento - Visualización de los datos (EDA)
* d) Preprocesamiento - Limpieza

**Objetivo principal:** Generar visualizaciones exploratorias que permitan entender distribuciones, tendencias y outliers. Posteriormente, realizar limpieza de nulos, duplicados e inconsistencias identificados en la etapa de comprensión.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, DateType, DecimalType, StringType

# Cargar las tablas originales (raw) para aplicar la limpieza
df_presupuestos = spark.table("workspace.nuevos_medios.presupuestos")
df_presupuesto_detalle = spark.table("workspace.nuevos_medios.presupuesto_detalle")
df_clientes = spark.table("workspace.nuevos_medios.clientes")
df_factura_cabecera = spark.table("workspace.nuevos_medios.factura_cabecera")
df_factura_detalle = spark.table("workspace.nuevos_medios.factura_detalle")
df_pago_proveedores = spark.table("workspace.nuevos_medios.pago_proveedores")
df_proveedores = spark.table("workspace.nuevos_medios.proveedores")
df_servicios = spark.table("workspace.nuevos_medios.servicios")
df_usuarios = spark.table("workspace.nuevos_medios.usuarios")

# Diccionario de dataframes "limpios" que se irán actualizando
cleaned_dfs = {
    "presupuestos": df_presupuestos,
    "presupuesto_detalle": df_presupuesto_detalle,
    "clientes": df_clientes,
    "factura_cabecera": df_factura_cabecera,
    "factura_detalle": df_factura_detalle,
    "pago_proveedores": df_pago_proveedores,
    "proveedores": df_proveedores,
    "servicios": df_servicios,
    "usuarios": df_usuarios
}

print("DataFrames cargados y listos para visualización y limpieza.")

DataFrames cargados y listos para visualización y limpieza.


## c) Preprocesamiento - Visualización (EDA)

**Objetivo:** Identificar visualmente outliers, tendencias y distribuciones clave que guiarán la limpieza. También se validarán duplicados de contenido (no solo de clave) identificados en el análisis del DOCX.

In [0]:
# El resumen estadístico (Bloque 2.3) mostró un 'Total' máximo de 669,768.91 y una media de 20,496.
# Esto sugiere una fuerte asimetría (skewness) y la presencia de outliers.

# En Databricks, al ejecutar display() sobre una columna numérica, se generan automáticamente un histograma y un boxplot.
print("Visualizando distribución de la columna 'Total' en df_presupuestos para detectar outliers.")
display(df_presupuestos.select("Total"))

Visualizando distribución de la columna 'Total' en df_presupuestos para detectar outliers.


Total
21809.0
7789.88
21809.0
703.13
6187.5
3516.0
3516.0
2200.0
2500.0
20490.7


Databricks visualization. Run in Databricks to view.

In [0]:
# Analizamos la frecuencia de presupuestos a lo largo del tiempo para entender la estacionalidad o tendencia del negocio.
print("Visualizando tendencia de presupuestos creados por fecha.")
display(df_presupuestos.groupBy("Fecha").count().orderBy("Fecha"))

Visualizando tendencia de presupuestos creados por fecha.


Fecha,count
2023-03-28,1
2023-04-06,11
2023-04-11,1
2023-05-16,1
2023-05-28,1
2023-06-10,1
2023-06-16,1
2023-06-20,6
2023-06-23,2
2023-06-24,1


Databricks visualization. Run in Databricks to view.

In [0]:
# Identificamos los clientes clave. Esto se alinea con el análisis de duplicados del DOCX (fuente 110).
print("Visualizando Top 20 Clientes (por IdCliente) según cantidad de presupuestos.")
display(df_presupuestos.groupBy("IdCliente").count().orderBy(F.desc("count")).limit(20))

Visualizando Top 20 Clientes (por IdCliente) según cantidad de presupuestos.


IdCliente,count
15,158
12,130
14,36
9,23
1,18
52,14
8,12
67,10
122,9
102,9


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#buscamos duplicados lógicos (mismo ítem en mismo presupuesto, o mismo pago).

print("Buscando líneas de detalle de presupuesto duplicadas (Mismo Presupuesto, Servicio y Precio)...")
display(
    df_presupuesto_detalle.groupBy("IdPresupuesto", "IdServicio", "PrecioUnitario")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print("Buscando pagos a proveedores duplicados (Mismo Proveedor, Monto y Fecha)...")
display(
    df_pago_proveedores.groupBy("IdProveedor", "Monto", "FechaPago")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

Buscando líneas de detalle de presupuesto duplicadas (Mismo Presupuesto, Servicio y Precio)...


IdPresupuesto,IdServicio,PrecioUnitario,count
671,2364,0.0,6
1758,2681,0.0,3
1749,2646,1500.0,2
1747,2612,1800.0,2
1749,2645,0.0,2
475,445,600.0,2
712,2552,0.56,2
475,2151,0.0,2
597,2216,5000.0,2
675,2424,250.0,2


Buscando pagos a proveedores duplicados (Mismo Proveedor, Monto y Fecha)...


IdProveedor,Monto,FechaPago,count
94,280.0,2023-12-20,3
141,55.0,2023-07-06,3
34,472.0,2023-11-06,3
57,6.6,2023-12-15,3
139,1240.0,2024-04-10,3
159,22.0,2024-07-18,3
85,176.0,2023-07-19,3
76,200.0,2024-03-04,3
57,6.6,2024-06-10,3
521,4000.0,2024-05-29,3


## d) Preprocesamiento - Limpieza

**Objetivo:** Aplicar las correcciones necesarias basadas en los hallazgos de las etapas (a), (b) y (c). Las acciones se centran en:
1.  **Inconsistencias de Tipos de Datos:** Corregir campos numéricos y fechas almacenados como `string` (DOCX, fuentes 3-22).
2.  **Manejo de Nulos:** Rellenar valores nulos en campos financieros críticos (`SubTotal`, `Monto`, etc.) con `0.0` para permitir cálculos.
3.  **Eliminación de Duplicados:** Eliminar duplicados de *contenido* (líneas de detalle o pagos idénticos) identificados en la visualización 4.
4.  **Integridad Referencial:** Eliminar registros "huérfanos" (ej. detalles sin cabecera) identificados en el notebook 01 (celdas 12-20).

In [0]:
print("Iniciando corrección de tipos de datos...")

# --- 1. df_presupuestos ---
# Hallazgos: Campos financieros y fechas como string. IdUsuario como double.
cleaned_dfs["presupuestos"] = df_presupuestos.withColumn(
    "SubTotal", F.col("SubTotal").cast(DecimalType(18, 2))
).withColumn(
    "Igv", F.col("Igv").cast(DecimalType(18, 2))
).withColumn(
    "SaldoInicial", F.col("SaldoInicial").cast(DecimalType(18, 2))
).withColumn(
    "SaldoActual", F.col("SaldoActual").cast(DecimalType(18, 2))
).withColumn(
    "SaldoFinal", F.col("SaldoFinal").cast(DecimalType(18, 2))
).withColumn( # Formato '2023-04-06'
    "FechaEntrega", F.to_date(F.col("FechaEntrega"), "yyyy-MM-dd")
).withColumn( # Formato '28/03/2023'
    "FechaCrea", F.to_date(F.col("FechaCrea"), "d/MM/yyyy")
).withColumn( # Corregir Key (DOCX, fuente 20)
    "IdUsuario", F.col("IdUsuario").cast(LongType())
)

# --- 2. df_presupuesto_detalle ---
# Hallazgos: Campos numéricos como string.
# Nota: 'Cantidad' y 'Fechas' tienen datos sucios ('Tilapia', '40 x 1'). Usamos cast() que convertirá los malformados a NULL (equivalente a try_cast).
cleaned_dfs["presupuesto_detalle"] = df_presupuesto_detalle.withColumn(
    "Cantidad", F.col("Cantidad").cast(DecimalType(18, 2))
).withColumn(
    "PrecioUnitario", F.col("PrecioUnitario").cast(DecimalType(18, 2))
).withColumn(
    "SubTotal", F.col("SubTotal").cast(DecimalType(18, 2))
).withColumn(
    "Fechas", F.col("Fechas").cast(DecimalType(10, 2)) # Columna sucia tratada como numérica
)

# --- 3. df_pago_proveedores ---
# Hallazgos: Campos financieros y fechas como string. Key de Proveedor como string.
cleaned_dfs["pago_proveedores"] = df_pago_proveedores.withColumn(
    "Monto", F.col("Monto").cast(DecimalType(18, 2))
).withColumn(
    "Itf", F.col("Itf").cast(DecimalType(18, 2))
).withColumn(
    "Igv", F.col("Igv").cast(DecimalType(18, 2))
).withColumn(
    "CostoTransferencia", F.col("CostoTransferencia").cast(DecimalType(18, 2))
).withColumn(
    "Detraccion", F.col("Detraccion").cast(DecimalType(18, 2))
).withColumn( # Formato '30/07/2024'
    "FechaPago", F.to_date(F.col("FechaPago"), "d/MM/yyyy")
).withColumn( # Formato '31/07/2024'
    "FechaDocReferencia", F.to_date(F.col("FechaDocReferencia"), "d/MM/yyyy")
).withColumn( # Corregir Key (DOCX, fuente 19)
    "IdProveedor", F.col("IdProveedor").cast(LongType())
)

print("Corrección de tipos de datos finalizada.")
print("--- Nuevo Esquema Presupuestos ---")
cleaned_dfs["presupuestos"].printSchema()
print("--- Nuevo Esquema Pago Proveedores ---")
cleaned_dfs["pago_proveedores"].printSchema()

Iniciando corrección de tipos de datos...
Corrección de tipos de datos finalizada.
--- Nuevo Esquema Presupuestos ---
root
 |-- IdPresupuesto: long (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- IdCliente: long (nullable = true)
 |-- Proyecto: string (nullable = true)
 |-- Lugar: string (nullable = true)
 |-- FechaEntrega: date (nullable = true)
 |-- TipoEvento: string (nullable = true)
 |-- Motivo: long (nullable = true)
 |-- Detalle: string (nullable = true)
 |-- IdEmpleado: string (nullable = true)
 |-- IdEstadoPresupuesto: string (nullable = true)
 |-- IdEjecutivo: string (nullable = true)
 |-- SubTotal: decimal(18,2) (nullable = true)
 |-- Igv: decimal(18,2) (nullable = true)
 |-- Total: double (nullable = true)
 |-- IdUsuario: long (nullable = true)
 |-- FechaCrea: date (nullable = true)
 |-- SaldoInicial: decimal(18,2) (nullable = true)
 |-- SaldoActual: decimal(18,2) (nullable = true)
 |-- SaldoFinal: decimal(18,2) (nullable = true)
 |-- OrdenDeCompra: string (nullab

In [0]:
print("Iniciando relleno de valores nulos en campos financieros...")

# Rellenamos con 0.0 los campos numéricos/financieros que quedaron nulos (sea por origen o por error de casting anterior)

cols_financieros_pres = ["SubTotal", "Igv", "Total", "SaldoInicial", "SaldoActual", "SaldoFinal", "Interes"]
cleaned_dfs["presupuestos"] = cleaned_dfs["presupuestos"].na.fill(0.0, subset=cols_financieros_pres)

cols_financieros_det = ["Cantidad", "PrecioUnitario", "SubTotal", "Fechas"]
cleaned_dfs["presupuesto_detalle"] = cleaned_dfs["presupuesto_detalle"].na.fill(0.0, subset=cols_financieros_det)

cols_financieros_pago = ["Monto", "Itf", "Igv", "CostoTransferencia", "Detraccion"]
cleaned_dfs["pago_proveedores"] = cleaned_dfs["pago_proveedores"].na.fill(0.0, subset=cols_financieros_pago)

print("Nulos en campos financieros rellenados con 0.0.")

Iniciando relleno de valores nulos en campos financieros...
Nulos en campos financieros rellenados con 0.0.


In [0]:
print("Iniciando eliminación de duplicados lógicos...")

# 1. Duplicados en presupuesto_detalle (basado en Visualización 4)
df = cleaned_dfs["presupuesto_detalle"]
registros_antes = df.count()
# Clave lógica: Mismo presupuesto, mismo servicio, mismo precio (DOCX, fuente 113)
cleaned_dfs["presupuesto_detalle"] = df.dropDuplicates(["IdPresupuesto", "IdServicio", "PrecioUnitario"])
registros_despues = cleaned_dfs["presupuesto_detalle"].count()
print(f"[Presupuesto Detalle] Registros antes: {registros_antes} | Registros después: {registros_despues} | Eliminados: {registros_antes - registros_despues}")

# 2. Duplicados en pago_proveedores (basado en Visualización 4)
df_pago = cleaned_dfs["pago_proveedores"]
registros_antes_pago = df_pago.count()
# Clave lógica: Mismo proveedor, monto, fecha y documento (DOCX, fuente 115 adaptada)
cleaned_dfs["pago_proveedores"] = df_pago.dropDuplicates(["IdProveedor", "Monto", "FechaPago", "DocReferencia"])
registros_despues_pago = cleaned_dfs["pago_proveedores"].count()
print(f"[Pago Proveedores] Registros antes: {registros_antes_pago} | Registros después: {registros_despues_pago} | Eliminados: {registros_antes_pago - registros_despues_pago}")

print("Eliminación de duplicados de contenido finalizada.")

Iniciando eliminación de duplicados lógicos...
[Presupuesto Detalle] Registros antes: 2134 | Registros después: 2080 | Eliminados: 54
[Pago Proveedores] Registros antes: 2161 | Registros después: 2126 | Eliminados: 35
Eliminación de duplicados de contenido finalizada.


In [0]:
print("Iniciando limpieza de registros huérfanos...")

# Generamos DFs de claves únicas (PKs) de las tablas maestras
pk_presupuestos = cleaned_dfs["presupuestos"].select("IdPresupuesto").distinct()
pk_factura_cab = cleaned_dfs["factura_cabecera"].select("idFacturaCab").distinct()
pk_servicios = cleaned_dfs["servicios"].select("IdServicio").distinct()
pk_clientes = cleaned_dfs["clientes"].select("IdCliente").distinct()
pk_proveedores = cleaned_dfs["proveedores"].select("IdProv").distinct()

# --- 1. Huérfanos en presupuesto_detalle (Notebook 01, celdas 12 y 20) ---
df = cleaned_dfs["presupuesto_detalle"]
registros_antes = df.count()
# Huérfano 1: 1 registro sin 'IdPresupuesto'
df = df.join(pk_presupuestos, "IdPresupuesto", "left_semi")
# Huérfano 2: 1 registro sin 'IdServicio'
df = df.join(pk_servicios, "IdServicio", "left_semi")
cleaned_dfs["presupuesto_detalle"] = df
registros_despues = df.count()
print(f"[Presupuesto Detalle] Eliminados {registros_antes - registros_despues} registros huérfanos.")

# --- 2. Huérfanos en factura_cabecera (Notebook 01, celdas 14 y 18) ---
df_fc = cleaned_dfs["factura_cabecera"]
registros_antes_fc = df_fc.count()
# Huérfano 1: 1 registro sin 'idPre' (IdPresupuesto)
df_fc = df_fc.join(pk_presupuestos, df_fc.idPre == pk_presupuestos.IdPresupuesto, "left_semi")
# Huérfano 2: 0 registros sin 'IdCliente' (Relación ya estaba OK)
df_fc = df_fc.join(pk_clientes, "idCliente", "left_semi")
cleaned_dfs["factura_cabecera"] = df_fc
registros_despues_fc = df_fc.count()
print(f"[Factura Cabecera] Eliminados {registros_antes_fc - registros_despues_fc} registros huérfanos.")

# --- 3. Huérfanos en factura_detalle (Notebook 01, celda 16) ---
df_fd = cleaned_dfs["factura_detalle"]
registros_antes_fd = df_fd.count()
# Huérfano 1: 6 registros sin 'idFacturaCab'
df_fd = df_fd.join(pk_factura_cab, "idFacturaCab", "left_semi")
cleaned_dfs["factura_detalle"] = df_fd
registros_despues_fd = df_fd.count()
print(f"[Factura Detalle] Eliminados {registros_antes_fd - registros_despues_fd} registros huérfanos.")

# --- 4. Huérfanos en pago_proveedores (Notebook 01, celda 19 - Error de casting) ---
# Ahora que los tipos están corregidos, podemos hacer el join.
df_pp = cleaned_dfs["pago_proveedores"]
registros_antes_pp = df_pp.count()
# Huérfanos por 'IdProveedor' que no existe en 'proveedores'
df_pp = df_pp.join(pk_proveedores, df_pp.IdProveedor == pk_proveedores.IdProv, "left_semi")
cleaned_dfs["pago_proveedores"] = df_pp
registros_despues_pp = df_pp.count()
print(f"[Pago Proveedores] Eliminados {registros_antes_pp - registros_despues_pp} pagos a proveedores desconocidos.")

print("Limpieza de integridad referencial finalizada.")

Iniciando limpieza de registros huérfanos...
[Presupuesto Detalle] Eliminados 1 registros huérfanos.
[Factura Cabecera] Eliminados 427 registros huérfanos.
[Factura Detalle] Eliminados 6 registros huérfanos.
[Pago Proveedores] Eliminados 176 pagos a proveedores desconocidos.
Limpieza de integridad referencial finalizada.


## Conclusión de la Etapa (c) y (d)

Las etapas de visualización y limpieza han sido completadas.

1.  **Visualización:** Se identificaron outliers en los montos de presupuestos, una tendencia temporal en la creación de los mismos, y se confirmó la existencia de duplicados de contenido (lógicos) en `presupuesto_detalle` y `pago_proveedores`.
2.  **Limpieza:**
    * Se corrigieron los tipos de datos (String a Numeric/Date/BigInt) en las tablas `presupuestos`, `presupuesto_detalle` y `pago_proveedores`, solucionando las inconsistencias de claves (DOCX, fuentes 19-20).
    * Se rellenaron los valores nulos en campos financieros críticos con `0.0`.
    * Se eliminaron los duplicados lógicos (líneas de pago y detalle repetidas).
    * Se eliminaron todos los registros huérfanos que violaban la integridad referencial.

Los DataFrames `cleaned_dfs` (o las tablas guardadas) están listos para la siguiente etapa (e) Transformación, a cargo de Nelly.

In [0]:
print("Guardando tablas limpias en el workspace...")

# Sobrescribimos las tablas en una nueva ubicación (o con un sufijo _limpio)
# para que la etapa (e) de Nelly pueda usarlas.

for nombre, df in cleaned_dfs.items():
    table_name = f"workspace.nuevos_medios.{nombre}_limpio"
    print(f"Guardando {table_name}...")
    # Usamos overwrite para asegurar que la celda se pueda re-ejecutar
    df.write.mode("overwrite").saveAsTable(table_name)

print("Proceso de visualización y limpieza completado. Tablas guardadas.")

Guardando tablas limpias en el workspace...
Guardando workspace.nuevos_medios.presupuestos_limpio...


---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-8005507387750693>, line 10
      8     print(f"Guardando {table_name}...")
      9     # Usamos overwrite para asegurar que la celda se pueda re-ejecutar
---> 10     df.write.mode("overwrite").saveAsTable(table_name)
     12 print("Proceso de visualización y limpieza completado. Tablas guardadas.")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1556, i